# Kindle Sequential Recommendation: Tuned SASRec and GRU4Rec

This notebook trains and evaluates:

1. **Popularity** - a non-personalised reference baseline.
2. **First-order Markov** - a simple sequential baseline using the last observed item.
3. **Tuned SASRec** - a causal Transformer for next-item prediction.
4. **GRU4Rec** - a recurrent next-item model for short histories.

The protocol is fixed throughout:

- interactions rated below 4 have already been removed by preprocessing;
- training uses only prefixes of `train_sequences.csv`;
- validation predicts `val.pos_idx` from the complete training sequence;
- testing predicts `test.pos_idx` from the training sequence **plus the validation item**;
- every target is ranked against its 100 fixed negatives;
- test targets never enter training inputs, training loss, model selection, or hyperparameter tuning;
- raw item indices start at 0, while model-side indices are shifted by +1 so that 0 remains padding.

With one relevant item per candidate set, `Recall@10` and `Hit@10` are numerically identical. Both labels are reported for compatibility. The offline candidate-set results do not measure catalogue-wide retrieval or post-deployment user impact.

## 1. Setup and configuration

Set `DATA_DIR` to the folder containing the nine prepared files. Defaults are deliberately small because the median training history is short. For a final run, keep the seed fixed and select epochs only with validation NDCG@10.

Required packages are `numpy`, `pandas`, and `torch`. If the environment does not provide them, install them in a separate setup cell (for example, `%pip install numpy pandas torch`) and restart the kernel before importing.

In [1]:
from __future__ import annotations

import ast
import copy
import json
import math
import random
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


@dataclass
class Config:
    data_dir: Path = Path("data")
    max_len: int = 50
    hidden_dim: int = 64
    num_blocks: int = 2
    num_heads: int = 2
    dropout: float = 0.20
    batch_size: int = 256
    learning_rate: float = 1e-3
    weight_decay: float = 0.0
    max_epochs: int = 30
    patience: int = 4
    eval_batch_size: int = 512
    num_workers: int = 0
    seed: int = 2026
    top_k: int = 10
    min_delta: float = 1e-5


CFG = Config()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(CFG.seed)
print("device:", DEVICE)
print(CFG)

device: cuda
Config(data_dir=WindowsPath('data'), max_len=50, hidden_dim=64, num_blocks=2, num_heads=2, dropout=0.2, batch_size=256, learning_rate=0.001, weight_decay=0.0, max_epochs=30, patience=4, eval_batch_size=512, num_workers=0, seed=2026, top_k=10, min_delta=1e-05)


## 2. Load files and normalise schemas

The loaders accept common column-name variants and sequence encodings such as JSON lists, Python-style lists, or delimiter-separated integers. Ambiguous or malformed schemas raise an error instead of guessing silently.

In [2]:
# Use the prepared Kindle data whether this notebook is run from proj2/ or the repository root.
DATA_DIR_CANDIDATES = (Path("processed_kindle"), Path("proj2/processed_kindle"))
CFG.data_dir = next((path for path in DATA_DIR_CANDIDATES if path.is_dir()), DATA_DIR_CANDIDATES[0])
print("data directory:", CFG.data_dir.resolve())

data directory: D:\unsw\COMP9727\proj2\processed_kindle


In [3]:
REQUIRED_FILES = [
    "train_sequences.csv", "train_triplets.csv", "val.csv", "test.csv",
    "val_negatives.csv", "test_negatives.csv", "item_metadata.csv",
    "preprocess_config.json", "preprocess_stats.json",
]


def require_files(data_dir: Path) -> None:
    missing = [name for name in REQUIRED_FILES if not (data_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing {len(missing)} required file(s) under {data_dir.resolve()}: {missing}"
        )


def choose_col(df: pd.DataFrame, candidates: Sequence[str], purpose: str) -> str:
    hits = [c for c in candidates if c in df.columns]
    if len(hits) != 1:
        raise ValueError(
            f"Expected exactly one {purpose} column from {list(candidates)}; "
            f"found {hits}. Available columns: {df.columns.tolist()}"
        )
    return hits[0]


def parse_int_list(value) -> List[int]:
    if isinstance(value, (list, tuple, np.ndarray)):
        values = list(value)
    elif pd.isna(value):
        values = []
    else:
        text = str(value).strip()
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            try:
                parsed = ast.literal_eval(text)
            except (ValueError, SyntaxError):
                parsed = [x for x in re.split(r"[\s,;|]+", text.strip("[]()")) if x]
        values = list(parsed) if isinstance(parsed, (list, tuple, np.ndarray)) else [parsed]
    try:
        return [int(x) for x in values]
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Cannot parse an integer list from {value!r}") from exc


def read_sequences(path: Path) -> Tuple[Dict[int, List[int]], pd.DataFrame]:
    df = pd.read_csv(path)
    user_col = choose_col(df, ["user_idx", "user_id", "user"], "user")
    seq_col = choose_col(df, ["item_seq", "sequence", "items"], "sequence")
    if df[user_col].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate user rows")
    sequences = {
        int(user): parse_int_list(seq)
        for user, seq in zip(df[user_col], df[seq_col])
    }
    return sequences, df


def read_targets(path: Path) -> Tuple[Dict[int, int], pd.DataFrame]:
    df = pd.read_csv(path)
    user_col = choose_col(df, ["user_idx", "user_id", "user"], "user")
    item_col = choose_col(df, ["pos_idx", "item_idx", "item_id", "item"], "positive item")
    if df[user_col].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate user rows")
    return dict(zip(df[user_col].astype(int), df[item_col].astype(int))), df


def read_negatives(path: Path) -> Tuple[Dict[int, List[int]], pd.DataFrame]:
    df = pd.read_csv(path)
    user_col = choose_col(df, ["user_idx", "user_id", "user"], "user")
    list_cols = [c for c in ["negatives", "negative_items", "neg_items", "neg_idx"] if c in df.columns]
    if list_cols:
        if len(list_cols) != 1 or df[user_col].duplicated().any():
            raise ValueError(f"Ambiguous or duplicate-row negative-list schema in {path.name}")
        result = {
            int(user): parse_int_list(items)
            for user, items in zip(df[user_col], df[list_cols[0]])
        }
    else:
        item_col = choose_col(
            df, ["negative_idx", "item_idx", "item_id", "item"], "negative item"
        )
        result = {
            int(user): group[item_col].astype(int).tolist()
            for user, group in df.groupby(user_col, sort=False)
        }
    return result, df


require_files(CFG.data_dir)
train_sequences, train_sequences_df = read_sequences(CFG.data_dir / "train_sequences.csv")
val_targets, val_df = read_targets(CFG.data_dir / "val.csv")
test_targets, test_df = read_targets(CFG.data_dir / "test.csv")
val_negatives, val_negatives_df = read_negatives(CFG.data_dir / "val_negatives.csv")
test_negatives, test_negatives_df = read_negatives(CFG.data_dir / "test_negatives.csv")
train_triplets_df = pd.read_csv(CFG.data_dir / "train_triplets.csv")
item_metadata_df = pd.read_csv(CFG.data_dir / "item_metadata.csv")

with open(CFG.data_dir / "preprocess_config.json", encoding="utf-8") as f:
    preprocess_config = json.load(f)
with open(CFG.data_dir / "preprocess_stats.json", encoding="utf-8") as f:
    preprocess_stats = json.load(f)

print("preprocess_config:", json.dumps(preprocess_config, indent=2, ensure_ascii=False))
print("preprocess_stats:", json.dumps(preprocess_stats, indent=2, ensure_ascii=False))
print(f"users={len(train_sequences):,}; train interactions={sum(map(len, train_sequences.values())):,}")

preprocess_config: {
  "reviews_path": "Kindle_Store.jsonl",
  "metadata_path": "meta_Kindle_Store.jsonl",
  "output_dir": "processed_kindle",
  "work_db_name": "preprocess_work.sqlite",
  "year_start": 2014,
  "year_end": 2022,
  "positive_threshold": 4.0,
  "deduplication": "keep_latest_user_item_interaction",
  "require_metadata_match": true,
  "min_user_positives": 5,
  "min_item_positives": 5,
  "filter_method": "iterative_5_core",
  "sample_strategy": "user_activity_stratified",
  "target_interactions": 1000000,
  "sample_overshoot_ratio": 1.62,
  "split": "chronological_leave_one_out",
  "ensure_eval_items_in_train": true,
  "default_num_negatives": 1,
  "negative_exclusion": "all_user_positive_items",
  "negative_unique_per_positive": true,
  "eval_num_negatives": 100,
  "missing_verified_purchase": false,
  "remove_categories": [
    "Kindle Store",
    "Kindle eBooks"
  ],
  "split_inner_category": true,
  "write_sequence_side_fields": true,
  "random_seed": 42,
  "progress_e

## 3. Protocol and leakage audit

These assertions verify the most consequential assumptions: aligned user sets, 100 distinct negatives, positive/negative separation, non-negative 0-based IDs, and absence of held-out positives from the training histories. A repeated item can be legitimate in raw review data, but under a leave-two-out item recommendation protocol it usually indicates a split or deduplication problem; inspect rather than bypass failures.

In [4]:
def audit_protocol() -> pd.DataFrame:
    users = set(train_sequences)
    sources = {
        "val targets": set(val_targets), "test targets": set(test_targets),
        "val negatives": set(val_negatives), "test negatives": set(test_negatives),
    }
    for name, source_users in sources.items():
        if source_users != users:
            raise AssertionError(
                f"User mismatch for {name}: missing={len(users-source_users)}, "
                f"extra={len(source_users-users)}"
            )

    all_ids = []
    failures = []
    for user in sorted(users):
        seq = train_sequences[user]
        vp, tp = val_targets[user], test_targets[user]
        vn, tn = val_negatives[user], test_negatives[user]
        all_ids.extend(seq + [vp, tp] + vn + tn)
        if len(vn) != 100 or len(set(vn)) != 100:
            failures.append((user, "validation negatives are not 100 distinct items"))
        if len(tn) != 100 or len(set(tn)) != 100:
            failures.append((user, "test negatives are not 100 distinct items"))
        if vp in vn or tp in tn:
            failures.append((user, "positive appears in its candidate negatives"))
        if vp in seq or tp in seq or tp == vp:
            failures.append((user, "held-out positive leaks/repeats across chronological splits"))
    if failures:
        raise AssertionError(f"Protocol audit failed; first examples: {failures[:10]}")
    if min(all_ids) < 0:
        raise AssertionError("Negative item index found")

    # Triplets are not used as an additional objective below, but their sampled labels
    # must still agree with the chronological train/holdout split.
    triplet_user_col = choose_col(train_triplets_df, ["user_idx", "user_id", "user"], "triplet user")
    triplet_pos_col = choose_col(
        train_triplets_df,
        ["pos_idx", "positive_idx", "pos_item_idx", "item_idx", "item_id", "item"],
        "triplet positive item",
    )
    triplet_neg_col = choose_col(
        train_triplets_df,
        ["neg_idx", "negative_idx", "neg_item_idx", "negative_item_idx"],
        "triplet negative item",
    )
    bad_triplets = []
    training_item_sets = {u: set(seq) for u, seq in train_sequences.items()}
    for row in train_triplets_df[[triplet_user_col, triplet_pos_col, triplet_neg_col]].itertuples(index=False, name=None):
        user, positive, negative = map(int, row)
        if user not in train_sequences:
            bad_triplets.append((user, positive, negative, "unknown user"))
            continue
        if positive not in training_item_sets[user]:
            bad_triplets.append((user, positive, negative, "positive is outside training sequence"))
        if positive == negative or negative in training_item_sets[user]:
            bad_triplets.append((user, positive, negative, "invalid/observed negative"))
        if negative in {val_targets[user], test_targets[user]}:
            bad_triplets.append((user, positive, negative, "negative is a held-out positive"))
        if len(bad_triplets) >= 10:
            break
    if bad_triplets:
        raise AssertionError(f"Triplet audit failed; first examples: {bad_triplets}")

    lengths = pd.Series([len(x) for x in train_sequences.values()], name="train_length")
    summary = lengths.describe(percentiles=[.25, .5, .75, .9, .95, .99]).to_frame()
    print("raw item ID range:", (min(all_ids), max(all_ids)))
    print("candidate protocol: 1 positive + 100 fixed negatives per user")
    return summary


audit_protocol()

raw item ID range: (0, 81321)
candidate protocol: 1 positive + 100 fixed negatives per user


,train_length
count,64867.000000
mean,13.629981
std,29.876118
min,3.000000
25%,4.000000
50%,6.000000
75%,12.000000
90%,26.000000
95%,45.000000
99%,143.000000


In [5]:
# Infer the embedding vocabulary from every split/candidate file, not metadata row count.
max_item_id = max(
    max(max(seq, default=-1) for seq in train_sequences.values()),
    max(val_targets.values()), max(test_targets.values()),
    max(max(x) for x in val_negatives.values()),
    max(max(x) for x in test_negatives.values()),
)
NUM_ITEMS = max_item_id + 1  # number of real, raw 0-based items
PAD_ID = 0                  # model-side padding ID


def to_model_id(raw_item_id: int) -> int:
    return int(raw_item_id) + 1


print(f"NUM_ITEMS={NUM_ITEMS:,}; embedding rows={NUM_ITEMS + 1:,} (including padding)")

NUM_ITEMS=81,322; embedding rows=81,323 (including padding)


## 4. Shared evaluation

The same evaluator is used for all models. Candidate order is deterministically shuffled per user before sorting, so tied scores do not systematically favour the positive item. MRR is computed over all 101 candidates; Hit, Recall, Precision and NDCG use the requested cutoff of 10. Because each candidate set has exactly one relevant item, Recall@10 equals Hit@10 and Precision@10 equals Hit@10 divided by 10. CandidateAccuracy@10 is reported only for compatibility with classification-style reporting: its 101-way class imbalance makes it much less informative than NDCG, Recall and MRR.

In [6]:
def rank_from_scores(
    user: int, candidates: Sequence[int], scores: Sequence[float], seed: int
) -> int:
    candidates = np.asarray(candidates, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if len(candidates) != len(scores):
        raise ValueError("Candidate and score lengths differ")
    rng = np.random.default_rng(seed + int(user))
    perm = rng.permutation(len(candidates))
    shuffled_scores = scores[perm]
    shuffled_positive_position = int(np.flatnonzero(perm == 0)[0])
    order = np.argsort(-shuffled_scores, kind="stable")
    return int(np.flatnonzero(order == shuffled_positive_position)[0]) + 1


def summarise_ranks(ranks: Sequence[int], k: int = 10, num_candidates: int = 101) -> Dict[str, float]:
    ranks = np.asarray(ranks, dtype=np.int64)
    hits = ranks <= k
    hit = float(hits.mean())
    if not 0 < k < num_candidates:
        raise ValueError("k must be between 1 and num_candidates - 1")
    # Classifying the Top-k as positive produces k predictions per 101-item set.
    # It is included only when a classification-style accuracy is explicitly required.
    candidate_accuracy = float((num_candidates - k + hit) / num_candidates)
    return {
        f"Hit@{k}": hit,
        f"Recall@{k}": hit,
        f"Precision@{k}": hit / k,
        f"CandidateAccuracy@{k}": candidate_accuracy,
        f"NDCG@{k}": float(np.where(hits, 1.0 / np.log2(ranks + 1), 0.0).mean()),
        "MRR": float((1.0 / ranks).mean()),
        "users": int(len(ranks)),
    }


def evaluate_score_function(
    contexts: Mapping[int, Sequence[int]],
    targets: Mapping[int, int],
    negatives: Mapping[int, Sequence[int]],
    score_fn,
    k: int = 10,
) -> Dict[str, float]:
    ranks = []
    for user in sorted(targets):
        candidates = [targets[user], *negatives[user]]
        scores = score_fn(user, contexts[user], candidates)
        ranks.append(rank_from_scores(user, candidates, scores, CFG.seed))
    return summarise_ranks(ranks, k)


val_contexts = train_sequences
test_contexts = {u: train_sequences[u] + [val_targets[u]] for u in train_sequences}

## 5. Baselines

Popularity counts only training interactions. The Markov model counts adjacent training transitions and uses global popularity as a small backoff/tie-breaker. It never learns from validation or test transitions.

In [7]:
popularity = np.zeros(NUM_ITEMS, dtype=np.int64)
transition_counts: Dict[int, Counter] = defaultdict(Counter)

for seq in train_sequences.values():
    for item in seq:
        popularity[item] += 1
    for previous, nxt in zip(seq[:-1], seq[1:]):
        transition_counts[previous][nxt] += 1

popularity_scaled = np.log1p(popularity) / max(np.log1p(popularity).max(), 1.0)


def popularity_score(user, context, candidates):
    return popularity_scaled[np.asarray(candidates)]


def markov_score(user, context, candidates):
    candidates = np.asarray(candidates)
    if not context:
        return popularity_scaled[candidates]
    outgoing = transition_counts.get(context[-1], {})
    transition = np.asarray([outgoing.get(int(item), 0) for item in candidates], dtype=float)
    return transition + 1e-3 * popularity_scaled[candidates]


baseline_rows = []
for name, scorer in [("Popularity", popularity_score), ("Markov-1", markov_score)]:
    val_metrics = evaluate_score_function(val_contexts, val_targets, val_negatives, scorer, CFG.top_k)
    test_metrics = evaluate_score_function(test_contexts, test_targets, test_negatives, scorer, CFG.top_k)
    baseline_rows.extend([
        {"model": name, "split": "validation", **val_metrics},
        {"model": name, "split": "test", **test_metrics},
    ])

baseline_results = pd.DataFrame(baseline_rows)
baseline_results

,model,split,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users
0,Popularity,validation,0.405645,0.405645,0.040565,0.905006,0.253269,0.224836,64867
1,Popularity,test,0.375121,0.375121,0.037512,0.904704,0.232436,0.206891,64867
2,Markov-1,validation,0.456827,0.456827,0.045683,0.905513,0.320822,0.295560,64867
3,Markov-1,test,0.417115,0.417115,0.041712,0.905120,0.288026,0.265202,64867


## 6. Prefix-to-next-item training data

For a training sequence `[i1, i2, i3]`, SASRec receives `[i1, i2]` and predicts `[i2, i3]` position-wise. One unobserved negative is sampled for each non-padding target. This is sampled binary cross-entropy, closely matching the original SASRec objective; `train_triplets.csv` is retained for preprocessing audit but is not mixed into a second, incompatible sampling scheme.

In [8]:
class SASRecTrainDataset(Dataset):
    def __init__(self, sequences: Mapping[int, Sequence[int]], num_items: int, max_len: int):
        self.users = [u for u, seq in sequences.items() if len(seq) >= 2]
        self.sequences = sequences
        self.num_items = num_items
        self.max_len = max_len
        self.seen = {u: set(seq) for u, seq in sequences.items()}

    def __len__(self) -> int:
        return len(self.users)

    def _negative(self, seen: set) -> int:
        if len(seen) >= self.num_items:
            raise RuntimeError("Cannot sample an unseen item for this user")
        item = np.random.randint(0, self.num_items)
        while item in seen:
            item = np.random.randint(0, self.num_items)
        return item

    def __getitem__(self, index: int):
        user = self.users[index]
        seq = self.sequences[user]
        raw_inputs = seq[:-1][-self.max_len:]
        raw_positives = seq[1:][-self.max_len:]
        n = len(raw_inputs)

        inputs = np.zeros(self.max_len, dtype=np.int64)
        positives = np.zeros(self.max_len, dtype=np.int64)
        negatives = np.zeros(self.max_len, dtype=np.int64)
        inputs[-n:] = np.asarray(raw_inputs) + 1
        positives[-n:] = np.asarray(raw_positives) + 1
        negatives[-n:] = np.asarray([self._negative(self.seen[user]) for _ in range(n)]) + 1
        return (
            torch.from_numpy(inputs),
            torch.from_numpy(positives),
            torch.from_numpy(negatives),
        )


train_dataset = SASRecTrainDataset(train_sequences, NUM_ITEMS, CFG.max_len)
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
)
print(f"training users with >=2 items: {len(train_dataset):,}")

training users with >=2 items: 64,867


In [9]:
# Keep held-out positives out of the sampled-negative pool.  They remain absent from
# training inputs and positive loss labels, but cannot be incorrectly labelled negative.
held_out_positives = {user: {val_targets[user], test_targets[user]} for user in train_sequences}
train_dataset.seen = {
    user: set(sequence).union(held_out_positives[user])
    for user, sequence in train_sequences.items()
}
print("negative-sampling exclusions include train, validation, and test positives")

negative-sampling exclusions include train, validation, and test positives


## 7. SASRec-lite model

The causal attention mask prevents every position from reading future items. Padding is masked separately. The final non-padding position represents the context for candidate scoring. The item embedding matrix is shared between sequence encoding and output scoring.

In [10]:
class SASRec(nn.Module):
    def __init__(
        self,
        num_items: int,
        max_len: int,
        hidden_dim: int = 64,
        num_blocks: int = 2,
        num_heads: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.max_len = max_len
        self.hidden_dim = hidden_dim
        self.item_embedding = nn.Embedding(num_items + 1, hidden_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_blocks)
        self.final_norm = nn.LayerNorm(hidden_dim)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def encode(self, item_ids: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = item_ids.shape
        if seq_len != self.max_len:
            raise ValueError(f"Expected sequence length {self.max_len}, got {seq_len}")
        positions = torch.arange(seq_len, device=item_ids.device).unsqueeze(0)
        x = self.item_embedding(item_ids) * math.sqrt(self.hidden_dim)
        x = self.dropout(x + self.position_embedding(positions))
        padding_mask = item_ids.eq(0)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=item_ids.device, dtype=torch.bool),
            diagonal=1,
        )
        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=padding_mask)
        return self.final_norm(x)

    def forward(self, item_ids: torch.Tensor) -> torch.Tensor:
        return self.encode(item_ids)

    def score_candidates(self, item_ids: torch.Tensor, candidate_ids: torch.Tensor) -> torch.Tensor:
        features = self.encode(item_ids)
        lengths = item_ids.ne(0).sum(dim=1).clamp_min(1)
        # Inputs are left padded, so the last array position is the final observed item.
        context_features = features[:, -1, :]
        candidate_embeddings = self.item_embedding(candidate_ids)
        return torch.einsum("bd,bcd->bc", context_features, candidate_embeddings)


model = SASRec(
    num_items=NUM_ITEMS,
    max_len=CFG.max_len,
    hidden_dim=CFG.hidden_dim,
    num_blocks=CFG.num_blocks,
    num_heads=CFG.num_heads,
    dropout=CFG.dropout,
).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

parameters: 5,307,968


C:\Users\al\AppData\Local\Temp\ipykernel_45112\1547733667.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_blocks)


## 8. Batched SASRec evaluation

Only validation metrics are used inside the training loop. The test evaluator is called once after restoring the best validation checkpoint.

In [11]:
def left_pad_context(context: Sequence[int], max_len: int) -> np.ndarray:
    context = list(context)[-max_len:]
    result = np.zeros(max_len, dtype=np.int64)
    if context:
        result[-len(context):] = np.asarray(context, dtype=np.int64) + 1
    return result


@torch.no_grad()
def evaluate_sasrec(
    model: SASRec,
    contexts: Mapping[int, Sequence[int]],
    targets: Mapping[int, int],
    negatives: Mapping[int, Sequence[int]],
    batch_size: int = 512,
    k: int = 10,
) -> Dict[str, float]:
    model.eval()
    users = sorted(targets)
    ranks = []
    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]
        input_np = np.stack([left_pad_context(contexts[u], CFG.max_len) for u in batch_users])
        candidate_np = np.asarray(
            [[targets[u], *negatives[u]] for u in batch_users], dtype=np.int64
        )
        input_tensor = torch.as_tensor(input_np, device=DEVICE)
        candidate_tensor = torch.as_tensor(candidate_np + 1, device=DEVICE)
        scores = model.score_candidates(input_tensor, candidate_tensor).cpu().numpy()
        for row, user in enumerate(batch_users):
            ranks.append(rank_from_scores(user, candidate_np[row], scores[row], CFG.seed))
    return summarise_ranks(ranks, k)

## 9. Train with validation NDCG@10 early stopping

Loss is averaged only over real target positions. The best in-memory state is selected by validation NDCG@10. For long experiments, optionally save `best_state` after training; do not select a checkpoint using test results.

In [12]:
def train_sasrec(model: SASRec):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay
    )
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    best_ndcg = -np.inf
    best_state = None
    stale_epochs = 0
    history = []

    for epoch in range(1, CFG.max_epochs + 1):
        started = time.time()
        model.train()
        total_loss = 0.0
        total_targets = 0

        for inputs, positives, negatives in train_loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            positives = positives.to(DEVICE, non_blocking=True)
            negatives = negatives.to(DEVICE, non_blocking=True)
            mask = positives.ne(0)

            optimizer.zero_grad(set_to_none=True)
            features = model(inputs)
            pos_logits = (features * model.item_embedding(positives)).sum(dim=-1)
            neg_logits = (features * model.item_embedding(negatives)).sum(dim=-1)
            losses = criterion(pos_logits, torch.ones_like(pos_logits))
            losses += criterion(neg_logits, torch.zeros_like(neg_logits))
            loss = losses[mask].mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            n_targets = int(mask.sum())
            total_loss += float(loss) * n_targets
            total_targets += n_targets

        val_metrics = evaluate_sasrec(
            model, val_contexts, val_targets, val_negatives,
            batch_size=CFG.eval_batch_size, k=CFG.top_k,
        )
        row = {
            "epoch": epoch,
            "train_loss": total_loss / total_targets,
            **val_metrics,
            "seconds": time.time() - started,
        }
        history.append(row)
        print(row)

        current = val_metrics[f"NDCG@{CFG.top_k}"]
        if current > best_ndcg + CFG.min_delta:
            best_ndcg = current
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= CFG.patience:
                print(f"Early stopping after epoch {epoch}; best validation NDCG@{CFG.top_k}={best_ndcg:.6f}")
                break

    if best_state is None:
        raise RuntimeError("Training produced no checkpoint")
    model.load_state_dict(best_state)
    return pd.DataFrame(history), best_state


# Training is run in the validation-only tuning section below.
# Keeping this helper separate prevents an untuned SASRec run from using the test set.

## 10. Final evaluation and comparison

Run the test evaluation only after architecture and training choices have been fixed from validation performance. Report the candidate-set protocol alongside every metric table; otherwise results are easily mistaken for full-catalogue ranking.

In [13]:
# The following tuning section selects all neural configurations from validation NDCG@10.
# Test metrics are calculated exactly once for each validation-selected architecture.

## 10. Validation-only neural tuning and GRU4Rec comparison

SASRec settings and GRU4Rec settings are trained on the same prefix-to-next-item task. Each trial is selected only by validation NDCG@10. After choosing the best configuration within each architecture, the two selected models are evaluated once on the untouched test split. GRU4Rec uses right padding and packed GRU sequences so padding cannot update its hidden state.

In [9]:
class GRU4RecTrainDataset(SASRecTrainDataset):
    """Prefix-to-next-item data with right padding for packed GRU processing."""
    def __getitem__(self, index: int):
        user = self.users[index]
        seq = self.sequences[user]
        raw_inputs = seq[:-1][-self.max_len:]
        raw_positives = seq[1:][-self.max_len:]
        n = len(raw_inputs)
        inputs = np.zeros(self.max_len, dtype=np.int64)
        positives = np.zeros(self.max_len, dtype=np.int64)
        negatives = np.zeros(self.max_len, dtype=np.int64)
        inputs[:n] = np.asarray(raw_inputs) + 1
        positives[:n] = np.asarray(raw_positives) + 1
        negatives[:n] = np.asarray([self._negative(self.seen[user]) for _ in range(n)]) + 1
        return torch.from_numpy(inputs), torch.from_numpy(positives), torch.from_numpy(negatives)


class MixedNegativeGRU4RecTrainDataset(GRU4RecTrainDataset):
    """Two negatives per target: one uniform and one popularity-weighted hard negative."""
    def __init__(self, sequences, num_items, max_len, num_negatives: int = 2):
        super().__init__(sequences, num_items, max_len)
        if num_negatives != 2:
            raise ValueError("This controlled experiment uses exactly two negatives")
        self.num_negatives = num_negatives
        weights = np.power(popularity.astype(np.float64) + 1.0, 0.75)
        popularity_probabilities = weights / weights.sum()
        # Sampling from this fixed weighted pool is much faster than invoking
        # np.random.choice(..., p=...) once for every sequence position.
        rng = np.random.default_rng(CFG.seed)
        self.popularity_pool = rng.choice(self.num_items, size=250_000, replace=True, p=popularity_probabilities)

    def _popular_negative(self, seen: set) -> int:
        item = int(self.popularity_pool[np.random.randint(len(self.popularity_pool))])
        while item in seen:
            item = int(self.popularity_pool[np.random.randint(len(self.popularity_pool))])
        return item

    def __getitem__(self, index: int):
        user = self.users[index]
        seq = self.sequences[user]
        raw_inputs, raw_positives = seq[:-1][-self.max_len:], seq[1:][-self.max_len:]
        n = len(raw_inputs)
        inputs = np.zeros(self.max_len, dtype=np.int64)
        positives = np.zeros(self.max_len, dtype=np.int64)
        negatives = np.zeros((self.max_len, self.num_negatives), dtype=np.int64)
        inputs[:n], positives[:n] = np.asarray(raw_inputs) + 1, np.asarray(raw_positives) + 1
        negatives[:n, 0] = np.asarray([self._negative(self.seen[user]) for _ in range(n)]) + 1
        negatives[:n, 1] = np.asarray([self._popular_negative(self.seen[user]) for _ in range(n)]) + 1
        return torch.from_numpy(inputs), torch.from_numpy(positives), torch.from_numpy(negatives)


class GRU4Rec(nn.Module):
    def __init__(self, num_items: int, max_len: int, hidden_dim: int = 64, num_layers: int = 1, dropout: float = 0.1):
        super().__init__()
        self.max_len = max_len
        self.hidden_dim = hidden_dim
        self.item_embedding = nn.Embedding(num_items + 1, hidden_dim, padding_idx=0)
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.GRU):
            for name, parameter in module.named_parameters():
                if "weight" in name:
                    nn.init.xavier_uniform_(parameter)
                else:
                    nn.init.zeros_(parameter)

    def forward(self, item_ids: torch.Tensor) -> torch.Tensor:
        lengths = item_ids.ne(0).sum(dim=1).clamp_min(1).cpu()
        embedded = self.dropout(self.item_embedding(item_ids))
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
        packed_outputs, _ = self.gru(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True, total_length=self.max_len)
        return self.dropout(outputs)

    def score_candidates(self, item_ids: torch.Tensor, candidate_ids: torch.Tensor) -> torch.Tensor:
        features = self(item_ids)
        final_positions = item_ids.ne(0).sum(dim=1).clamp_min(1).sub(1)
        context_features = features[torch.arange(item_ids.size(0), device=item_ids.device), final_positions]
        return torch.einsum("bd,bcd->bc", context_features, self.item_embedding(candidate_ids))


def right_pad_context(context: Sequence[int], max_len: int) -> np.ndarray:
    context = list(context)[-max_len:]
    result = np.zeros(max_len, dtype=np.int64)
    if context:
        result[:len(context)] = np.asarray(context, dtype=np.int64) + 1
    return result


@torch.no_grad()
def evaluate_neural(model, contexts, targets, negatives, max_len: int, pad_context, batch_size: int = 512, k: int = 10):
    model.eval()
    ranks = []
    users = sorted(targets)
    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]
        inputs = np.stack([pad_context(contexts[user], max_len) for user in batch_users])
        candidates = np.asarray([[targets[user], *negatives[user]] for user in batch_users], dtype=np.int64)
        scores = model.score_candidates(torch.as_tensor(inputs, device=DEVICE), torch.as_tensor(candidates + 1, device=DEVICE)).cpu().numpy()
        ranks.extend(rank_from_scores(user, candidates[row], scores[row], CFG.seed) for row, user in enumerate(batch_users))
    return summarise_ranks(ranks, k)


In [15]:
def make_train_loader(dataset_cls, max_len: int, **dataset_kwargs):
    dataset = dataset_cls(train_sequences, NUM_ITEMS, max_len, **dataset_kwargs)
    dataset.seen = {user: set(sequence).union(held_out_positives[user]) for user, sequence in train_sequences.items()}
    return DataLoader(dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=torch.cuda.is_available())


def train_neural(model, loader, max_len: int, pad_context, learning_rate: float, weight_decay: float, max_epochs: int, patience: int, label: str, loss_name: str = "bce"):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    best_ndcg, best_state, stale_epochs = -np.inf, None, 0
    history = []
    for epoch in range(1, max_epochs + 1):
        started, total_loss, total_targets = time.time(), 0.0, 0
        model.train()
        for inputs, positives, negatives in loader:
            inputs, positives, negatives = (tensor.to(DEVICE, non_blocking=True) for tensor in (inputs, positives, negatives))
            mask = positives.ne(0)
            optimizer.zero_grad(set_to_none=True)
            features = model(inputs)
            pos_logits = (features * model.item_embedding(positives)).sum(dim=-1)
            neg_embeddings = model.item_embedding(negatives)
            neg_logits = (features.unsqueeze(-2) * neg_embeddings).sum(dim=-1) if negatives.ndim == 3 else (features * neg_embeddings).sum(dim=-1)
            if loss_name == "bpr":
                if neg_logits.ndim == 2:
                    neg_logits = neg_logits.unsqueeze(-1)
                loss = torch.nn.functional.softplus(neg_logits - pos_logits.unsqueeze(-1)).mean(dim=-1)[mask].mean()
            elif loss_name == "bce":
                negative_loss = criterion(neg_logits, torch.zeros_like(neg_logits))
                if negative_loss.ndim == 3:
                    negative_loss = negative_loss.mean(dim=-1)
                loss = (criterion(pos_logits, torch.ones_like(pos_logits)) + negative_loss)[mask].mean()
            else:
                raise ValueError(f"Unknown loss_name={loss_name!r}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            n_targets = int(mask.sum())
            total_loss += loss.detach().item() * n_targets
            total_targets += n_targets
        val_metrics = evaluate_neural(model, val_contexts, val_targets, val_negatives, max_len, pad_context, CFG.eval_batch_size, CFG.top_k)
        row = {"model": label, "epoch": epoch, "train_loss": total_loss / total_targets, **val_metrics, "seconds": time.time() - started}
        history.append(row)
        print(row)
        current = val_metrics[f"NDCG@{CFG.top_k}"]
        if current > best_ndcg + CFG.min_delta:
            best_ndcg, best_state, stale_epochs = current, copy.deepcopy(model.state_dict()), 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"{label}: early stopping at epoch {epoch}; best validation NDCG@{CFG.top_k}={best_ndcg:.6f}")
                break
    model.load_state_dict(best_state)
    return pd.DataFrame(history), {name: value.detach().cpu().clone() for name, value in best_state.items()}


SASREC_TRIALS = [
    {"name": "SASRec-short-20", "max_len": 20, "hidden_dim": 64, "num_blocks": 2, "num_heads": 2, "dropout": 0.10, "learning_rate": 1e-3},
]
GRU4REC_TRIALS = [
    {"name": "GRU4Rec-short-10", "max_len": 10, "hidden_dim": 64, "num_layers": 1, "dropout": 0.10, "learning_rate": 1e-3},
    {"name": "GRU4Rec-short-20", "max_len": 20, "hidden_dim": 64, "num_layers": 1, "dropout": 0.10, "learning_rate": 1e-3},
    {"name": "GRU4Rec-short-20-low-lr", "max_len": 20, "hidden_dim": 64, "num_layers": 1, "dropout": 0.10, "learning_rate": 5e-4},
    {"name": "GRU4Rec-short-20-mixedneg-bpr", "max_len": 20, "hidden_dim": 64, "num_layers": 1, "dropout": 0.10, "learning_rate": 1e-3, "dataset_cls": MixedNegativeGRU4RecTrainDataset, "loss_name": "bpr", "num_negatives": 2, "max_epochs": 60},
]
TUNING_EPOCHS, TUNING_PATIENCE, TUNING_WEIGHT_DECAY = 100, 10, 1e-5
SEED_STABILITY_SEEDS = (2026, 2027, 2028)


In [16]:
def run_sasrec_trial(spec):
    seed_everything(CFG.seed)
    loader = make_train_loader(SASRecTrainDataset, spec["max_len"])
    model = SASRec(NUM_ITEMS, spec["max_len"], spec["hidden_dim"], spec["num_blocks"], spec["num_heads"], spec["dropout"]).to(DEVICE)
    history, state = train_neural(model, loader, spec["max_len"], left_pad_context, spec["learning_rate"], TUNING_WEIGHT_DECAY, TUNING_EPOCHS, TUNING_PATIENCE, spec["name"])
    result = {**spec, "architecture": "SASRec", "history": history, "state": state, "parameters": sum(parameter.numel() for parameter in model.parameters())}
    result["validation"] = history.loc[history[f"NDCG@{CFG.top_k}"].idxmax(), [f"Hit@{CFG.top_k}", f"Recall@{CFG.top_k}", f"Precision@{CFG.top_k}", f"CandidateAccuracy@{CFG.top_k}", f"NDCG@{CFG.top_k}", "MRR", "epoch"]].to_dict()
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result


def run_gru4rec_trial(spec, seed: int = CFG.seed):
    seed_everything(seed)
    dataset_cls = spec.get("dataset_cls", GRU4RecTrainDataset)
    loader = make_train_loader(dataset_cls, spec["max_len"], **({"num_negatives": spec["num_negatives"]} if "num_negatives" in spec else {}))
    model = GRU4Rec(NUM_ITEMS, spec["max_len"], spec["hidden_dim"], spec["num_layers"], spec["dropout"]).to(DEVICE)
    history, state = train_neural(model, loader, spec["max_len"], right_pad_context, spec["learning_rate"], TUNING_WEIGHT_DECAY, spec.get("max_epochs", TUNING_EPOCHS), TUNING_PATIENCE, spec["name"], spec.get("loss_name", "bce"))
    result = {**spec, "architecture": "GRU4Rec", "seed": seed, "history": history, "state": state, "parameters": sum(parameter.numel() for parameter in model.parameters())}
    result["validation"] = history.loc[history[f"NDCG@{CFG.top_k}"].idxmax(), [f"Hit@{CFG.top_k}", f"Recall@{CFG.top_k}", f"Precision@{CFG.top_k}", f"CandidateAccuracy@{CFG.top_k}", f"NDCG@{CFG.top_k}", "MRR", "epoch"]].to_dict()
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result


sasrec_trials = [run_sasrec_trial(spec) for spec in SASREC_TRIALS]
gru4rec_trials = [run_gru4rec_trial(spec) for spec in GRU4REC_TRIALS]
all_trials = sasrec_trials + gru4rec_trials
tuning_validation_df = pd.DataFrame([{key: (value.__name__ if key == "dataset_cls" else value) for key, value in trial.items() if key not in {"history", "state", "validation"}} | trial["validation"] for trial in all_trials])
tuning_validation_df.sort_values(f"NDCG@{CFG.top_k}", ascending=False)

C:\Users\al\AppData\Local\Temp\ipykernel_45112\1547733667.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_blocks)


{'model': 'SASRec-short-20', 'epoch': 1, 'train_loss': 1.2759256489489401, 'Hit@10': 0.12595002081181494, 'Recall@10': 0.12595002081181494, 'NDCG@10': 0.061854932657149614, 'MRR': 0.06517440891011472, 'users': 64867, 'seconds': 6.982460260391235}
{'model': 'SASRec-short-20', 'epoch': 2, 'train_loss': 1.0625843244936604, 'Hit@10': 0.13510721938736184, 'Recall@10': 0.13510721938736184, 'NDCG@10': 0.06758319701412561, 'MRR': 0.06989792306233811, 'users': 64867, 'seconds': 5.86835789680481}
{'model': 'SASRec-short-20', 'epoch': 3, 'train_loss': 0.8935595211696183, 'Hit@10': 0.14303112522546133, 'Recall@10': 0.14303112522546133, 'NDCG@10': 0.0732296176284385, 'MRR': 0.07461675251344778, 'users': 64867, 'seconds': 5.92006778717041}
{'model': 'SASRec-short-20', 'epoch': 4, 'train_loss': 0.7449604687108645, 'Hit@10': 0.14711640741825582, 'Recall@10': 0.14711640741825582, 'NDCG@10': 0.07661439817820759, 'MRR': 0.07760387272599634, 'users': 64867, 'seconds': 5.823467254638672}
{'model': 'SASRec-

,name,max_len,hidden_dim,num_blocks,num_heads,dropout,learning_rate,architecture,parameters,Hit@10,Recall@10,NDCG@10,MRR,epoch,num_layers,seed
2,GRU4Rec-short-20,20,64,NaN,NaN,0.1,0.0010,GRU4Rec,5229632,0.677725,0.677725,0.455477,0.400238,99,1.0,2026.0
3,GRU4Rec-short-20-low-lr,20,64,NaN,NaN,0.1,0.0005,GRU4Rec,5229632,0.641497,0.641497,0.421944,0.369237,100,1.0,2026.0
1,GRU4Rec-short-10,10,64,NaN,NaN,0.1,0.0010,GRU4Rec,5229632,0.624169,0.624169,0.412942,0.362483,99,1.0,2026.0
0,SASRec-short-20,20,64,2.0,2.0,0.1,0.0010,SASRec,5306048,0.169824,0.169824,0.098172,0.098114,47,NaN,NaN


In [17]:
# Repeat only the validation-selected GRU configuration across seeds.
# These runs quantify stability; they do not use the test set.
best_gru_for_stability = max(gru4rec_trials, key=lambda trial: trial["validation"][f"NDCG@{CFG.top_k}"])
seed_stability_trials = [best_gru_for_stability] + [
    run_gru4rec_trial(best_gru_for_stability, seed=seed)
    for seed in SEED_STABILITY_SEEDS if seed != best_gru_for_stability["seed"]
]
seed_stability_df = pd.DataFrame([{
    "seed": trial["seed"], "epoch": trial["validation"]["epoch"],
    f"NDCG@{CFG.top_k}": trial["validation"][f"NDCG@{CFG.top_k}"],
    f"Recall@{CFG.top_k}": trial["validation"][f"Recall@{CFG.top_k}"],
    f"Precision@{CFG.top_k}": trial["validation"][f"Precision@{CFG.top_k}"],
    f"CandidateAccuracy@{CFG.top_k}": trial["validation"][f"CandidateAccuracy@{CFG.top_k}"],
    "MRR": trial["validation"]["MRR"],
} for trial in seed_stability_trials])
seed_stability_summary = seed_stability_df.drop(columns=["seed", "epoch"]).agg(["mean", "std"])
display(seed_stability_df)
display(seed_stability_summary)

{'model': 'GRU4Rec-short-20', 'epoch': 1, 'train_loss': 1.2995715444381686, 'Hit@10': 0.4059537206900273, 'Recall@10': 0.4059537206900273, 'NDCG@10': 0.24891042511563582, 'MRR': 0.220034742594839, 'users': 64867, 'seconds': 7.454541921615601}
{'model': 'GRU4Rec-short-20', 'epoch': 2, 'train_loss': 1.1587812205395296, 'Hit@10': 0.43943761851172397, 'Recall@10': 0.43943761851172397, 'NDCG@10': 0.26640495512171253, 'MRR': 0.23318914239233315, 'users': 64867, 'seconds': 7.296975612640381}
{'model': 'GRU4Rec-short-20', 'epoch': 3, 'train_loss': 1.0856845564950595, 'Hit@10': 0.46934496739482323, 'Recall@10': 0.46934496739482323, 'NDCG@10': 0.28518727398099725, 'MRR': 0.2486897275773254, 'users': 64867, 'seconds': 7.227041721343994}
{'model': 'GRU4Rec-short-20', 'epoch': 4, 'train_loss': 1.0242033883743837, 'Hit@10': 0.49007970154315755, 'Recall@10': 0.49007970154315755, 'NDCG@10': 0.300296738232478, 'MRR': 0.2618661372639495, 'users': 64867, 'seconds': 7.415994882583618}
{'model': 'GRU4Rec-s

,seed,epoch,NDCG@10,Hit@10,MRR
0,2026,99,0.455477,0.677725,0.400238
1,2027,100,0.453646,0.675582,0.398623
2,2028,100,0.454071,0.676723,0.398748


,NDCG@10,Hit@10,MRR
mean,0.454398,0.676677,0.399203
std,0.000958,0.001072,0.000898


In [18]:
def restore_selected(trial):
    if trial["architecture"] == "SASRec":
        selected = SASRec(NUM_ITEMS, trial["max_len"], trial["hidden_dim"], trial["num_blocks"], trial["num_heads"], trial["dropout"]).to(DEVICE)
        pad_context = left_pad_context
    else:
        selected = GRU4Rec(NUM_ITEMS, trial["max_len"], trial["hidden_dim"], trial["num_layers"], trial["dropout"]).to(DEVICE)
        pad_context = right_pad_context
    selected.load_state_dict(trial["state"])
    return selected, pad_context


best_sasrec = max(sasrec_trials, key=lambda trial: trial["validation"][f"NDCG@{CFG.top_k}"])
best_gru4rec = max(gru4rec_trials, key=lambda trial: trial["validation"][f"NDCG@{CFG.top_k}"])
selected_trials = [best_sasrec, best_gru4rec]
neural_rows, selected_models = [], {}
for trial in selected_trials:
    selected_model, pad_context = restore_selected(trial)
    selected_models[trial["architecture"]] = selected_model
    val_metrics = evaluate_neural(selected_model, val_contexts, val_targets, val_negatives, trial["max_len"], pad_context, CFG.eval_batch_size, CFG.top_k)
    test_metrics = evaluate_neural(selected_model, test_contexts, test_targets, test_negatives, trial["max_len"], pad_context, CFG.eval_batch_size, CFG.top_k)
    neural_rows.extend([{"model": trial["name"], "split": "validation", **val_metrics}, {"model": trial["name"], "split": "test", **test_metrics}])
results_df = pd.concat([baseline_results, pd.DataFrame(neural_rows)], ignore_index=True)
history_df, best_state = best_sasrec["history"], best_sasrec["state"]
# GRU4Rec is the selected production model: it wins validation selection and is the
# only neural model used by the qualitative and full-catalogue inference cells below.
final_trial, final_model, final_pad_context = best_gru4rec, selected_models["GRU4Rec"], right_pad_context
results_df.sort_values(["split", f"NDCG@{CFG.top_k}"], ascending=[True, False])

C:\Users\al\AppData\Local\Temp\ipykernel_45112\1547733667.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_blocks)


,model,split,Hit@10,Recall@10,NDCG@10,MRR,users
7,GRU4Rec-short-20,test,0.640742,0.640742,0.423697,0.371434,64867
3,Markov-1,test,0.417115,0.417115,0.288026,0.265202,64867
1,Popularity,test,0.375121,0.375121,0.232436,0.206891,64867
5,SASRec-short-20,test,0.166741,0.166741,0.094416,0.094408,64867
6,GRU4Rec-short-20,validation,0.677725,0.677725,0.455477,0.400238,64867
2,Markov-1,validation,0.456827,0.456827,0.320822,0.295560,64867
0,Popularity,validation,0.405645,0.405645,0.253269,0.224836,64867
4,SASRec-short-20,validation,0.169824,0.169824,0.098172,0.098114,64867


### Presentation-only checkpoint restore

Use this cell after the data, baseline, and GRU definition cells when presenting existing results. It restores the validation-selected GRU4Rec checkpoint without rerunning neural training, then refreshes the fixed-protocol comparison and the ensemble cells below. Full training and the mixed-negative ablation remain in the preceding tuning section.


In [10]:
# Restore the saved validation-selected GRU4Rec for a fast, reproducible presentation run.
checkpoint_path = Path("outputs/gru4rec_best.pt")
if not checkpoint_path.exists():
    raise FileNotFoundError("Run the tuning section once before using presentation-only restore.")
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
final_trial = checkpoint["config"]
final_model = GRU4Rec(NUM_ITEMS, final_trial["max_len"], final_trial["hidden_dim"], final_trial["num_layers"], final_trial["dropout"]).to(DEVICE)
final_model.load_state_dict(checkpoint["model_state_dict"])
final_pad_context = right_pad_context
results_path = Path("outputs/candidate_set_results.csv")
results_df = pd.read_csv(results_path) if results_path.exists() else baseline_results.copy()
# The ensemble cell appends a newly evaluated row, so remove any stale copy first.
results_df = results_df[~results_df["model"].astype(str).str.startswith("GRU4Rec-Markov")].copy()
display(results_df.sort_values(["split", f"NDCG@{CFG.top_k}"], ascending=[True, False]))


,model,split,Hit@10,Recall@10,NDCG@10,MRR,users,Precision@10,CandidateAccuracy@10
7,GRU4Rec-short-20,test,0.640742,0.640742,0.423697,0.371434,64867.0,0.064074,0.907334
3,Markov-1,test,0.417115,0.417115,0.288026,0.265202,64867.0,0.041712,0.905120
1,Popularity,test,0.375121,0.375121,0.232436,0.206891,64867.0,0.037512,0.904704
5,SASRec-short-20,test,0.166741,0.166741,0.094416,0.094408,64867.0,0.016674,0.902641
6,GRU4Rec-short-20,validation,0.677725,0.677725,0.455477,0.400238,64867.0,0.067773,0.907700
2,Markov-1,validation,0.456827,0.456827,0.320822,0.295560,64867.0,0.045683,0.905513
0,Popularity,validation,0.405645,0.405645,0.253269,0.224836,64867.0,0.040565,0.905006
4,SASRec-short-20,validation,0.169824,0.169824,0.098172,0.098114,64867.0,0.016982,0.902672


## 11. Low-complexity sequential refinement: GRU4Rec-Markov ensemble

The GRU captures a multi-item sequential state, while Markov-1 retains a direct next-item transition signal from the final item. Candidate scores are standardised per user and the GRU weight is selected only by validation NDCG@10. This adds no trainable parameters and only one sparse transition lookup per candidate at inference. The test split is evaluated once after selection.


In [11]:
def standardise_candidate_scores(scores: np.ndarray) -> np.ndarray:
    mean = scores.mean(axis=1, keepdims=True)
    std = scores.std(axis=1, keepdims=True)
    return (scores - mean) / np.maximum(std, 1e-8)


@torch.no_grad()
def evaluate_gru_markov_ensemble(model, contexts, targets, negatives, max_len: int, gru_weight: float, batch_size: int = 512, k: int = 10):
    model.eval()
    ranks = []
    users = sorted(targets)
    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]
        inputs = np.stack([right_pad_context(contexts[user], max_len) for user in batch_users])
        candidates = np.asarray([[targets[user], *negatives[user]] for user in batch_users], dtype=np.int64)
        gru_scores = model.score_candidates(torch.as_tensor(inputs, device=DEVICE), torch.as_tensor(candidates + 1, device=DEVICE)).cpu().numpy()
        markov_scores = np.vstack([markov_score(user, contexts[user], candidates[row]) for row, user in enumerate(batch_users)])
        scores = gru_weight * standardise_candidate_scores(gru_scores) + (1.0 - gru_weight) * standardise_candidate_scores(markov_scores)
        ranks.extend(rank_from_scores(user, candidates[row], scores[row], CFG.seed) for row, user in enumerate(batch_users))
    return summarise_ranks(ranks, k)


ENSEMBLE_GRU_WEIGHTS = (0.50, 0.65, 0.75, 0.85, 1.00)
ensemble_validation_rows = []
for weight in ENSEMBLE_GRU_WEIGHTS:
    metrics = evaluate_gru_markov_ensemble(final_model, val_contexts, val_targets, val_negatives, final_trial["max_len"], weight, CFG.eval_batch_size, CFG.top_k)
    ensemble_validation_rows.append({"gru_weight": weight, "markov_weight": 1.0 - weight, **metrics})
ensemble_validation_df = pd.DataFrame(ensemble_validation_rows).sort_values(f"NDCG@{CFG.top_k}", ascending=False)
best_ensemble = ensemble_validation_df.iloc[0].to_dict()
best_ensemble_test = evaluate_gru_markov_ensemble(final_model, test_contexts, test_targets, test_negatives, final_trial["max_len"], best_ensemble["gru_weight"], CFG.eval_batch_size, CFG.top_k)
ensemble_name = f"GRU4Rec-Markov (GRU={best_ensemble['gru_weight']:.2f})"
results_df = pd.concat([results_df, pd.DataFrame([
    {"model": ensemble_name, "split": "validation", **{key: value for key, value in best_ensemble.items() if key not in {"gru_weight", "markov_weight"}}},
    {"model": ensemble_name, "split": "test", **best_ensemble_test},
])], ignore_index=True)
display(ensemble_validation_df)
results_df.sort_values(["split", f"NDCG@{CFG.top_k}"], ascending=[True, False])


,gru_weight,markov_weight,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users
3,0.85,0.15,0.676708,0.676708,0.067671,0.907690,0.474027,0.424634,64867
2,0.75,0.25,0.661045,0.661045,0.066104,0.907535,0.460903,0.412848,64867
4,1.00,0.00,0.677725,0.677725,0.067773,0.907700,0.455477,0.400238,64867
1,0.65,0.35,0.637998,0.637998,0.063800,0.907307,0.440750,0.394364,64867
0,0.50,0.50,0.592690,0.592690,0.059269,0.906858,0.406686,0.365021,64867


,model,split,Hit@10,Recall@10,NDCG@10,MRR,users,Precision@10,CandidateAccuracy@10
9,GRU4Rec-Markov (GRU=0.85),test,0.634776,0.634776,0.437167,0.390772,64867.0,0.063478,0.907275
7,GRU4Rec-short-20,test,0.640742,0.640742,0.423697,0.371434,64867.0,0.064074,0.907334
3,Markov-1,test,0.417115,0.417115,0.288026,0.265202,64867.0,0.041712,0.905120
1,Popularity,test,0.375121,0.375121,0.232436,0.206891,64867.0,0.037512,0.904704
5,SASRec-short-20,test,0.166741,0.166741,0.094416,0.094408,64867.0,0.016674,0.902641
8,GRU4Rec-Markov (GRU=0.85),validation,0.676708,0.676708,0.474027,0.424634,64867.0,0.067671,0.907690
6,GRU4Rec-short-20,validation,0.677725,0.677725,0.455477,0.400238,64867.0,0.067773,0.907700
2,Markov-1,validation,0.456827,0.456827,0.320822,0.295560,64867.0,0.045683,0.905513
0,Popularity,validation,0.405645,0.405645,0.253269,0.224836,64867.0,0.040565,0.905006
4,SASRec-short-20,validation,0.169824,0.169824,0.098172,0.098114,64867.0,0.016982,0.902672


In [12]:
# Optional reproducibility outputs.
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_DIR / "candidate_set_results.csv", index=False)
if "history_df" in globals(): history_df.to_csv(OUTPUT_DIR / "sasrec_training_history.csv", index=False)
if "best_gru4rec" in globals(): best_gru4rec["history"].to_csv(OUTPUT_DIR / "gru4rec_training_history.csv", index=False)
if "tuning_validation_df" in globals(): tuning_validation_df.to_csv(OUTPUT_DIR / "neural_tuning_validation_results.csv", index=False)
if "seed_stability_df" in globals(): seed_stability_df.to_csv(OUTPUT_DIR / "gru4rec_seed_stability.csv", index=False)
if "ensemble_validation_df" in globals(): ensemble_validation_df.to_csv(OUTPUT_DIR / "gru_markov_ensemble_validation.csv", index=False)
if "best_state" in globals() and "best_sasrec" in globals(): torch.save(
    {
        "model_state_dict": best_state,
        "config": {key: value for key, value in best_sasrec.items() if key not in {"history", "state", "validation"}},
        "num_items": NUM_ITEMS,
        "raw_to_model_item_offset": 1,
    },
    OUTPUT_DIR / "sasrec_lite_best.pt",
)
if "best_gru4rec" in globals(): torch.save(
    {
        "model_state_dict": best_gru4rec["state"],
        "config": {key: value for key, value in best_gru4rec.items() if key not in {"history", "state", "validation", "dataset_cls"}},
        "num_items": NUM_ITEMS,
        "raw_to_model_item_offset": 1,
    },
    OUTPUT_DIR / "gru4rec_best.pt",
)
print("saved outputs to", OUTPUT_DIR.resolve())

saved outputs to D:\unsw\COMP9727\proj2\outputs


## 11. Optional qualitative inspection with metadata

This section is for interpretation only. Metadata does not enter SASRec training. It displays the target and the model's top-ranked candidates for selected users, which can reveal popularity bias, repeated authors/categories, or implausible recommendations that aggregate metrics hide.

In [12]:
def infer_metadata_item_col(df: pd.DataFrame) -> str:
    return choose_col(df, ["item_idx", "item_id", "item"], "metadata item")


@torch.no_grad()
def inspect_user(user: int, split: str = "test", top_n: int = 10) -> pd.DataFrame:
    if split == "validation":
        contexts, targets, negatives = val_contexts, val_targets, val_negatives
    elif split == "test":
        contexts, targets, negatives = test_contexts, test_targets, test_negatives
    else:
        raise ValueError("split must be 'validation' or 'test'")
    candidates = np.asarray([targets[user], *negatives[user]], dtype=np.int64)
    inputs = torch.as_tensor(final_pad_context(contexts[user], final_trial["max_len"])[None, :], device=DEVICE)
    candidate_tensor = torch.as_tensor(candidates[None, :] + 1, device=DEVICE)
    scores = final_model.score_candidates(inputs, candidate_tensor)[0].cpu().numpy()
    order = np.argsort(-scores, kind="stable")[:top_n]
    ranked = pd.DataFrame({
        "rank": np.arange(1, len(order) + 1),
        "item_idx": candidates[order],
        "score": scores[order],
        "is_target": candidates[order] == targets[user],
    })
    metadata_item_col = infer_metadata_item_col(item_metadata_df)
    metadata = item_metadata_df.rename(columns={metadata_item_col: "item_idx"})
    keep = [c for c in ["item_idx", "title", "author", "categories"] if c in metadata.columns]
    return ranked.merge(metadata[keep], on="item_idx", how="left")


# Example after training:
# inspect_user(user=sorted(test_targets)[0], split="test", top_n=10)

## 12. Category-aware Top-10 inference and catalogue behaviour

The final GRU4Rec scores unseen catalogue items. Candidate filtering retains books sharing a user-profile category; MMR reranking then discourages repeated authors and highly overlapping category labels. These presentation-time controls do not affect the fixed-candidate accuracy evaluation.

In [13]:
metadata_item_col = infer_metadata_item_col(item_metadata_df)
metadata_for_ranking = item_metadata_df.rename(columns={metadata_item_col: "item_idx"}).set_index("item_idx", drop=True)


def category_set(item: int) -> frozenset:
    if item not in metadata_for_ranking.index:
        return frozenset()
    value = metadata_for_ranking.at[item, "categories"] if "categories" in metadata_for_ranking.columns else ""
    return frozenset(part.strip() for part in str(value).split(";") if part.strip())


ITEM_CATEGORIES = {item: category_set(item) for item in range(NUM_ITEMS)}
ITEM_AUTHORS = {item: str(metadata_for_ranking.at[item, "author"]).strip() if item in metadata_for_ranking.index and "author" in metadata_for_ranking.columns else "" for item in range(NUM_ITEMS)}


def user_category_profile(context: Sequence[int], max_categories: int = 5) -> set:
    counts = Counter(category for item in context for category in ITEM_CATEGORIES[item])
    return {category for category, _ in counts.most_common(max_categories)}


@torch.no_grad()
def score_unseen_catalogue(user: int, context: Optional[Sequence[int]] = None, category_filter: bool = False, candidate_batch_size: int = 4096) -> pd.DataFrame:
    context = list(train_sequences[user] if context is None else context)
    if not context:
        raise ValueError("At least one observed item is required")
    seen, profile = set(context), user_category_profile(context)
    candidates = np.fromiter((item for item in range(NUM_ITEMS) if item not in seen and (not category_filter or not profile or bool(ITEM_CATEGORIES[item] & profile))), dtype=np.int64)
    if len(candidates) == 0:
        raise RuntimeError("No unseen candidates after filtering")
    inputs = torch.as_tensor(final_pad_context(context, final_trial["max_len"])[None, :], device=DEVICE)
    final_model.eval()
    scores = []
    for start in range(0, len(candidates), candidate_batch_size):
        batch = candidates[start:start + candidate_batch_size]
        scores.append(final_model.score_candidates(inputs, torch.as_tensor(batch[None, :] + 1, device=DEVICE))[0].cpu().numpy())
    ranked = pd.DataFrame({"item_idx": candidates, "score": np.concatenate(scores)})
    return ranked.sort_values("score", ascending=False, kind="stable").reset_index(drop=True)


def item_similarity(left: int, right: int) -> float:
    left_categories, right_categories = ITEM_CATEGORIES[left], ITEM_CATEGORIES[right]
    category_similarity = len(left_categories & right_categories) / max(len(left_categories | right_categories), 1)
    author_similarity = float(bool(ITEM_AUTHORS[left]) and ITEM_AUTHORS[left] == ITEM_AUTHORS[right])
    return max(category_similarity, author_similarity)


def mmr_rerank(scored_items: pd.DataFrame, top_k: int = 10, pool_size: int = 200, relevance_weight: float = 0.85) -> pd.DataFrame:
    pool = scored_items.head(pool_size).copy().reset_index(drop=True)
    score_range = max(pool.score.max() - pool.score.min(), 1e-12)
    pool["relevance"] = (pool.score - pool.score.min()) / score_range
    selected = []
    while len(selected) < min(top_k, len(pool)):
        remaining = [index for index in pool.index if index not in selected]
        choice = max(remaining, key=lambda index: relevance_weight * pool.at[index, "relevance"] - (1 - relevance_weight) * max((item_similarity(int(pool.at[index, "item_idx"]), int(pool.at[chosen, "item_idx"])) for chosen in selected), default=0.0))
        selected.append(choice)
    result = pool.loc[selected, ["item_idx", "score"]].copy()
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    return result


def recommend_top_k(user: int, context: Optional[Sequence[int]] = None, top_k: int = 10, category_filter: bool = True, mmr: bool = True) -> pd.DataFrame:
    scored = score_unseen_catalogue(user, context, category_filter)
    ranked = mmr_rerank(scored, top_k=top_k) if mmr else scored.head(top_k).assign(rank=lambda frame: np.arange(1, len(frame) + 1))[["rank", "item_idx", "score"]]
    keep = [column for column in ["title", "author", "categories", "average_rating", "rating_number"] if column in metadata_for_ranking.columns]
    return ranked.merge(metadata_for_ranking[keep].reset_index(), on="item_idx", how="left")


def recommendation_diagnostics(users: Sequence[int], top_k: int = 10) -> pd.DataFrame:
    catalogue_items, rows = set(), []
    total_popularity = popularity.sum()
    for user in users:
        recommendations = recommend_top_k(user, top_k=top_k)
        items = recommendations.item_idx.astype(int).tolist()
        catalogue_items.update(items)
        similarities = [item_similarity(left, right) for position, left in enumerate(items) for right in items[position + 1:]]
        novelty = np.mean([-np.log2((popularity[item] + 1) / (total_popularity + NUM_ITEMS)) for item in items])
        rows.append({"user_idx": user, "novelty": novelty, "intra_list_diversity": 1 - np.mean(similarities) if similarities else 0.0, "avg_train_popularity": float(np.mean(popularity[items]))})
    return pd.DataFrame(rows), {"users": len(users), "catalogue_coverage": len(catalogue_items) / NUM_ITEMS, "unique_recommended_items": len(catalogue_items)}


# Example: recommend_top_k(user=0)
# diagnostics_df, catalogue_metrics = recommendation_diagnostics(sorted(train_sequences)[:100])

In [14]:
# Deterministic system-level diagnostics on a manageable user sample.
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
diagnostic_rng = np.random.default_rng(CFG.seed)
diagnostic_users = diagnostic_rng.choice(np.asarray(sorted(train_sequences)), size=min(50, len(train_sequences)), replace=False).tolist()
catalogue_diagnostics_df, catalogue_metrics = recommendation_diagnostics(diagnostic_users)
catalogue_diagnostics_df.to_csv(OUTPUT_DIR / "gru4rec_catalogue_diagnostics.csv", index=False)
with open(OUTPUT_DIR / "gru4rec_catalogue_metrics.json", "w", encoding="utf-8") as handle:
    json.dump(catalogue_metrics, handle, indent=2)
print("catalogue metrics:", catalogue_metrics)
display(catalogue_diagnostics_df.describe())

catalogue metrics: {'users': 50, 'catalogue_coverage': 0.004623595091119254, 'unique_recommended_items': 376}


,user_idx,novelty,intra_list_diversity,avg_train_popularity
count,50.000000,50.000000,50.000000,50.000000
mean,31745.180000,13.682728,0.284259,146.628000
std,17928.297753,1.639647,0.228892,127.627652
min,832.000000,11.158217,0.000000,7.700000
25%,17678.000000,12.091477,0.000000,34.725000
50%,30252.000000,13.589245,0.259259,87.200000
75%,46484.750000,15.231056,0.508333,254.350000
max,62696.000000,16.916948,0.688889,434.900000


## 12. Report checklist and interpretation boundaries

Record the following in the project report:

- preprocessing threshold (`rating >= 4`), 5-core procedure, chronological leave-two-out split, and fixed negative sampling, copied exactly from the JSON files;
- dataset counts and sequence-length distribution after preprocessing;
- all model hyperparameters, random seed, maximum epochs, patience, and the epoch selected by validation NDCG@10;
- Popularity, Markov-1, and SASRec results under the same 101-item candidate sets;
- the fact that Hit@10 equals Recall@10 because there is exactly one relevant item;
- test context includes the validation item, while neither held-out test items nor candidate negatives enter training;
- limitations: sampled-negative metrics are not full-catalogue metrics, offline logs reflect exposure/selection bias, short histories restrict the advantage of deeper attention, and results may vary across seeds.

A defensible conclusion is conditional: SASRec provides evidence of value from attention only if it beats the sequential Markov baseline consistently on validation and the untouched test split. Beating popularity alone is insufficient to isolate the benefit of attention.